# Analysis walkthrough — *Self-monitoring in perceptual decisions by humans and machines*

This notebook demonstrates every analysis in the manuscript by calling the
reusable modules in `scripts_analysis/` and `core/`. It is organised by manuscript figure:

| Section | Manuscript | What it shows |
|---|---|---|
| 1. Simulation | **Fig 1** (+ Supp Fig 1) | Bias-blind vs. bias-aware confidence readout |
| 2. Experiment 1 | **Fig 4** | Negative direct effect of bias on confidence (4- & 8-choice) |
| 3. Experiment 2 (joint) | **Fig 5** | Speed vs. accuracy focus, fit *together* (Bias x Condition) |
| 4. Standard vs. metacognitive ANN | **Fig 6 / Fig 7** | Positive (standard) vs. negative (metacognitive) bias effect |
| 5. Metacognitive module variants | **Supp** | Chen-style, dual-output, distribution-shift |

Sections that need external data (behavioral CSVs on OSF, ANN result CSVs from the
`scripts_ann/` scripts) are **guarded**: if a file is missing the cell prints where
to get it and moves on. The simulation (Section 1) needs no external data.

## Section 0 — Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repo root, so `scripts_analysis` / `core` import

import numpy as np
import pandas as pd

DATA = "../data/human_data"      # preprocess.py outputs (*_aggregated.csv)
MODEL = "../data/model_data"     # ANN per-image CSVs (scripts_ann/*)
print("Python:", sys.version.split()[0])

## Section 1 — Generative SDT simulation (Figure 1)

A bias-blind confidence readout produces a **positive** accuracy-controlled bias
coefficient; a bias-aware readout produces a **negative** one, and higher
metacognitive sensitivity (Phi). This runs a reduced-size simulation for speed — use
the defaults (`n_subj=200, n_trial=1600`) to reproduce the manuscript values, and
`sim.plot_figure1(results, "figure1.pdf")` for the figure.

In [ ]:
from scripts_analysis import simulation as sim

results = sim.run_simulation(k_list=(4, 8), n_subj=60, n_trial=1000, seed=1, n_boot=300)
sim.print_summary(results)

# Robustness grid (Supplementary Figure 1):
# blind, aware, sig, cn = sim.run_robustness_sweep()
# sim.plot_sweep(blind, aware, sig, cn, "supp_fig1_sweep.pdf")

## Section 2 — Experiment 1 regression & mediation (Figure 4)

Each Experiment-1 condition is analysed separately. Run `preprocess.py` first
(README section 5.2 Step 1) to create the `*_aggregated.csv` files.

In [ ]:
from scripts_analysis.aggregate import build_human_frame
from scripts_analysis.regression import get_mixed_model_coefficients_random
from scripts_analysis.mediation import run_mixed_effect_mediation, summarize_mediation

def run_exp1(name, csv):
    if not os.path.exists(csv):
        print(f"[skip] {name}: missing {csv} -- run preprocess.py (README 5.2 Step 1).")
        return
    df = build_human_frame(pd.read_csv(csv))
    print(f"\n=== {name}  (n_subjects={df['subject_idx'].nunique()}) ===")
    coeffs, models = get_mixed_model_coefficients_random(
        df, "confidence_z", ["FAR_z", "accuracy_z", "rt_z"],
        target_regressor="FAR_z", print_summary=False)
    for i, m in models.items():
        b = m.params.get("FAR_z", float("nan"))
        print(f"  Model {i+1}: bias->confidence beta = {b:+.3f}")
    trace = run_mixed_effect_mediation(df, "FAR_z", "accuracy_z", "confidence_z")
    print(summarize_mediation(trace)[["parameter", "mean", "significance"]].to_string(index=False))

run_exp1("Exp 1 -- 8-choice", f"{DATA}/Experiment1_8_choice_aggregated.csv")
run_exp1("Exp 1 -- 4-choice", f"{DATA}/Experiment1_4_choice_aggregated.csv")

## Section 3 — Experiment 2, joint moderated analysis (Figure 5)

The two speed-accuracy-tradeoff conditions are fit **together**, with condition as
an effect-coded moderator, so the **Bias x Condition** interaction is tested
directly. The residual negative bias->confidence effect is present under accuracy
focus and eliminated under speed focus.

In [ ]:
from scripts_analysis.aggregate import build_combined_exp2_frame
from scripts_analysis.regression import get_moderated_regression, add_effect_coded_condition
from scripts_analysis.mediation import run_moderated_mediation, summarize_moderated_mediation

pa = f"{DATA}/Experiment2_accuracy_aggregated.csv"
ps = f"{DATA}/Experiment2_speed_aggregated.csv"
if os.path.exists(pa) and os.path.exists(ps):
    df = build_combined_exp2_frame(pd.read_csv(pa), pd.read_csv(ps))

    mod = get_moderated_regression(df, bias_var="FAR_z", outcome_var="confidence_z")
    print("\nBias x Condition interaction across nested models:")
    print(mod["interaction"].to_string(index=False))
    print("\nPer-condition simple slopes (accuracy focus vs. speed focus):")
    print(mod["simple_slopes"].to_string(index=False))

    df = add_effect_coded_condition(df)
    trace = run_moderated_mediation(df, predictor="FAR_z")
    print("\nJoint moderated mediation:")
    print(summarize_moderated_mediation(trace).to_string(index=False))
else:
    print("Missing Experiment 2 aggregated CSVs -- run preprocess.py for both conditions "
          "(README 5.2 Step 1).")

## Section 4 — Standard vs. metacognitive ANN (Figures 6 & 7)

The ANN CSVs are aggregated to the digit level and analysed with the *same*
functions used for humans. The confidence column selects the readout:
`conf_top2diff` = standard ANN (**Fig 6**, positive), `conf_meta` = metacognitive
learned head (**Fig 7**, negative). Generate the CSVs first with
`scripts_ann/test_metacognitive.py` (README section 5.3 Step 3).

In [ ]:
from scripts_analysis.aggregate import aggregate_ann_csv

def run_ann(name, csv, conf_col):
    if not os.path.exists(csv):
        print(f"[skip] {name}: missing {csv} -- see README 5.3.")
        return
    df = aggregate_ann_csv(csv, conf_col=conf_col)
    coeffs, models = get_mixed_model_coefficients_random(
        df, "confidence_z", ["FAR_z", "accuracy_z"],       # ANNs have no RT
        target_regressor="FAR_z", print_summary=False)
    b = [m.params.get("FAR_z", float("nan")) for m in models.values()]
    print(f"{name}: bias->confidence beta  (bias only) {b[0]:+.3f}   (+accuracy) {b[1]:+.3f}")

meta_csv = f"{MODEL}/meta/alexnet_fixed_base.csv"
run_ann("AlexNet -- standard (Fig 6)",           meta_csv, "conf_top2diff")
run_ann("AlexNet -- metacognitive head (Fig 7)", meta_csv, "conf_meta")

## Section 5 — Metacognitive module variants (Supplementary)

All four metacognitive modules keep the base classifier frozen, so FAR and accuracy
are identical across modules — only the confidence readout differs. Point each row
at the CSV produced by the corresponding `scripts_ann` script/mode.

In [ ]:
variants = [
    ("fixed_base  (main, Fig 7)", f"{MODEL}/meta/alexnet_fixed_base.csv",  "conf_meta"),
    ("chen        (Supp 6.1)",    f"{MODEL}/meta/alexnet_chen.csv",        "conf_meta"),
    ("dual        (Supp 6.2)",    f"{MODEL}/meta/alexnet_dual.csv",        "conf_meta"),
    ("dist-shift  (Supp 6.3)",    f"{MODEL}/distribution_shift/alexnet_distribution_shift.csv", "conf_meta"),
]
for name, csv, col in variants:
    run_ann(f"AlexNet -- {name}", csv, col)

---
**Notes**

- Reduced sizes are used here for speed; use the module defaults to reproduce the
  manuscript numbers.
- Bayesian models use NUTS with `target_accept=0.95`; expect a few minutes per fit.
- See `README.md` section 5 for the full command-line reproduction steps, including
  ANN training and the M-SDT robustness analyses (Supp Figs 2-3).